# 📊 MODULE 4: ADVANCED TIME SERIES MODELS
## Time Series Analytics - Educational Notebook for Google Colab

---

### Topics Covered:
- **4.1** Seasonal ARIMA (SARIMA)
- **4.2** Simple Exponential Smoothing
- **4.3** Holt's Linear Trend Method
- **4.4** Holt-Winters Seasonal Method
- **4.5** Vector Autoregression (VAR) Model
- **4.6** ARCH Effect Detection
- **4.7** GARCH Model for Volatility
- **4.8** Comprehensive Model Comparison

---

**Instructions:** Run each cell in order. Each topic section is self-contained after the setup cell.

## 🔧 Setup: Install and Import Required Packages

Run this cell first to install dependencies and download data.

In [ ]:
# Install required packages (uncomment if running on Colab for the first time)
# !pip install yfinance arch --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import het_arch
from scipy import stats

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All packages imported successfully!")

In [ ]:
# Download data
print("Downloading data...")
btc = yf.download('BTC-USD', start='2020-01-01', progress=False)['Close']
gold = yf.download('GC=F', start='2020-01-01', progress=False)['Close']
mrf = yf.download('MRF.NS', start='2020-01-01', progress=False)['Close']

# Clean data
btc = btc.dropna()
gold = gold.dropna()
mrf = mrf.dropna()

print(f"✓ Bitcoin: {len(btc)} observations")
print(f"✓ Gold: {len(gold)} observations")
print(f"✓ MRF: {len(mrf)} observations")
print("\n✅ Setup complete! Ready to run Module 4 topics.")

---

# 📌 TOPIC 4.1: Seasonal ARIMA (SARIMA)

### Key Concepts:
SARIMA extends ARIMA to handle **seasonal patterns** in data.

**SARIMA(p, d, q)(P, D, Q, s)**:
- $(p, d, q)$ = Non-seasonal order
- $(P, D, Q, s)$ = Seasonal order with period $s$

**Formula:**
$$(1-\phi L)(1-\Phi L^s)(1-L)(1-L^s)Y_t = (1+\theta L)(1+\Theta L^s)\epsilon_t$$

Where:
- $\phi$ = Non-seasonal AR coefficient
- $\Phi$ = Seasonal AR coefficient
- $\theta$ = Non-seasonal MA coefficient
- $\Theta$ = Seasonal MA coefficient
- $s$ = Seasonal period (e.g., 12 for monthly data)

In [ ]:
# Use monthly aggregated Bitcoin data
btc_monthly = btc.resample('ME').mean()
train_btc_monthly = btc_monthly[:int(len(btc_monthly)*0.85)]

print(f"Monthly Bitcoin data: {len(btc_monthly)} months")
print(f"Training data: {len(train_btc_monthly)} months")

In [ ]:
# Fit SARIMA(1,1,1)(1,1,1,12) model
# (p,d,q) = non-seasonal order
# (P,D,Q,s) = seasonal order with s=12 (monthly seasonality)
sarima_model = SARIMAX(train_btc_monthly, 
                       order=(1, 1, 1),
                       seasonal_order=(1, 1, 1, 12))
sarima_fitted = sarima_model.fit(disp=False)

print("SARIMA(1,1,1)(1,1,1,12) Model Summary:")
print(f"  Non-seasonal AR (φ): {sarima_fitted.params['ar.L1']:.4f}")
print(f"  Non-seasonal MA (θ): {sarima_fitted.params['ma.L1']:.4f}")
print(f"  Seasonal AR (Φ): {sarima_fitted.params['ar.S.L12']:.4f}")
print(f"  Seasonal MA (Θ): {sarima_fitted.params['ma.S.L12']:.4f}")
print(f"  AIC: {sarima_fitted.aic:.2f}")

In [ ]:
# Forecast next 12 months
forecast_sarima = sarima_fitted.forecast(steps=12)
forecast_sarima_ci = sarima_fitted.get_forecast(steps=12).conf_int()
forecast_sarima_index = pd.date_range(start=train_btc_monthly.index[-1] + pd.DateOffset(months=1), 
                                      periods=12, freq='ME')

# Plot with seasonal pattern visible
plt.figure(figsize=(14, 6))
plt.plot(train_btc_monthly.index[-24:], train_btc_monthly[-24:], 
         label='Training Data (Last 24 months)', color='blue', linewidth=2, marker='o')
plt.plot(forecast_sarima_index, forecast_sarima, 
         label='SARIMA Forecast', color='red', linewidth=2, marker='s', markersize=8)
plt.fill_between(forecast_sarima_index, forecast_sarima_ci.iloc[:, 0], forecast_sarima_ci.iloc[:, 1],
                 color='red', alpha=0.2)
plt.title('SARIMA Model: Bitcoin Monthly Forecast with Seasonality', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Bitcoin Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- SARIMA captures **both trend and seasonal patterns**
- Use when data shows repeating patterns at fixed intervals (monthly, quarterly, yearly)
- The seasonal period $s$ should match the data frequency (s=12 for monthly, s=4 for quarterly)

---

# 📌 TOPIC 4.2: Simple Exponential Smoothing

### Key Concepts:
Simple Exponential Smoothing (SES) creates forecasts using a **weighted average** of past observations.

**Formula:**
$$\hat{Y}_{t+1} = \alpha Y_t + (1-\alpha)\hat{Y}_t$$

Where:
- $\alpha$ = Smoothing parameter (0 < α < 1)
- Higher $\alpha$ → More weight to recent observations (faster response)
- Lower $\alpha$ → More weight to historical observations (smoother)

In [ ]:
# Use Gold price data
train_gold = gold[:int(len(gold)*0.8)]

# Try different smoothing levels
alphas = [0.1, 0.5, 0.9]
forecasts = {}

for alpha in alphas:
    ses_model = SimpleExpSmoothing(train_gold)
    ses_fitted = ses_model.fit(smoothing_level=alpha, optimized=False)
    forecasts[alpha] = ses_fitted.fittedvalues

print(f"Fitted SES models with α = {alphas}")

In [ ]:
# Plot all three with original data
plt.figure(figsize=(14, 6))
plt.plot(train_gold.index[-200:], train_gold[-200:], 
         label='Actual Data', color='black', linewidth=1.5, alpha=0.7)
plt.plot(train_gold.index[-200:], forecasts[0.1][-200:], 
         label='α=0.1 (Slow response)', color='blue', linewidth=2)
plt.plot(train_gold.index[-200:], forecasts[0.5][-200:], 
         label='α=0.5 (Moderate)', color='green', linewidth=2)
plt.plot(train_gold.index[-200:], forecasts[0.9][-200:], 
         label='α=0.9 (Fast response)', color='red', linewidth=2)
plt.title('Simple Exponential Smoothing: Effect of Alpha Parameter', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Gold Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Effect of Alpha (α) on Smoothing:")
print("="*50)
print("  α=0.1: Heavy smoothing, slow to react to changes")
print("  α=0.5: Balanced approach")
print("  α=0.9: Light smoothing, quick to react to changes")
print("\n✓ Higher α → More weight to recent observations")

### 📝 Key Takeaway:
- SES is best for data with **no trend and no seasonality**
- The smoothing parameter $\alpha$ controls the trade-off between noise reduction and responsiveness
- All future forecasts from SES are the same value (flat line)

---

# 📌 TOPIC 4.3: Holt's Linear Trend Method

### Key Concepts:
Holt's method extends SES to capture **linear trends** in data.

**Two Equations:**

$$\text{Level: } \ell_t = \alpha Y_t + (1-\alpha)(\ell_{t-1} + b_{t-1})$$

$$\text{Trend: } b_t = \beta(\ell_t - \ell_{t-1}) + (1-\beta)b_{t-1}$$

**Forecast:**
$$\hat{Y}_{t+h} = \ell_t + h \cdot b_t$$

Where:
- $\alpha$ = Level smoothing parameter
- $\beta$ = Trend smoothing parameter
- $h$ = Forecast horizon

In [ ]:
# Use MRF data with clear trend
train_mrf = mrf[:int(len(mrf)*0.8)]

# Apply Holt's linear method
holt_model = Holt(train_mrf)
holt_fitted = holt_model.fit(optimized=True)

# Apply simple exponential smoothing for comparison
ses_model_comp = SimpleExpSmoothing(train_mrf)
ses_fitted_comp = ses_model_comp.fit(optimized=True)

print(f"Holt's Method - α (level): {holt_fitted.params['smoothing_level']:.4f}")
print(f"Holt's Method - β (trend): {holt_fitted.params['smoothing_trend']:.4f}")
print(f"SES - α: {ses_fitted_comp.params['smoothing_level']:.4f}")

In [ ]:
# Forecast 30 days
forecast_holt = holt_fitted.forecast(steps=30)
forecast_ses_comp = ses_fitted_comp.forecast(steps=30)
forecast_index = pd.date_range(start=train_mrf.index[-1] + pd.Timedelta(days=1), 
                               periods=30, freq='D')

# Plot comparison
plt.figure(figsize=(14, 6))
plt.plot(train_mrf.index[-100:], train_mrf[-100:], 
         label='Training Data', color='blue', linewidth=2)
plt.plot(forecast_index, forecast_holt, 
         label="Holt's Linear Trend", color='red', linewidth=2, marker='o')
plt.plot(forecast_index, forecast_ses_comp, 
         label='Simple Exponential Smoothing', color='green', linewidth=2, marker='s')
plt.title("Holt's Linear Trend vs Simple Exponential Smoothing", fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('MRF Price (INR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- Holt's method captures **linear trends** that SES cannot
- SES produces flat forecasts; Holt's produces sloped forecasts
- Use when data shows a consistent upward or downward trend

---

# 📌 TOPIC 4.4: Holt-Winters Seasonal Method

### Key Concepts:
Holt-Winters extends Holt's method to capture **both trend and seasonality**.

**Two Types:**
1. **Additive**: Seasonal variations are constant
   - $Y_t = \text{Trend} + \text{Seasonal} + \text{Error}$

2. **Multiplicative**: Seasonal variations grow with the level
   - $Y_t = \text{Trend} \times \text{Seasonal} \times \text{Error}$

**When to use which:**
- Additive: When seasonal fluctuations are roughly constant
- Multiplicative: When seasonal fluctuations grow proportionally with the level

In [ ]:
# Use Bitcoin data (weekly aggregated for seasonality)
btc_weekly = btc.resample('W').mean()
train_btc_weekly = btc_weekly[:int(len(btc_weekly)*0.85)]

print(f"Weekly Bitcoin data: {len(btc_weekly)} weeks")
print(f"Training data: {len(train_btc_weekly)} weeks")

In [ ]:
# Apply Holt-Winters with additive seasonal
hw_add = ExponentialSmoothing(train_btc_weekly, 
                              trend='add', 
                              seasonal='add', 
                              seasonal_periods=52)  # 52 weeks in a year
hw_add_fitted = hw_add.fit(optimized=True)

# Apply Holt-Winters with multiplicative seasonal
hw_mul = ExponentialSmoothing(train_btc_weekly, 
                              trend='add', 
                              seasonal='mul', 
                              seasonal_periods=52)
hw_mul_fitted = hw_mul.fit(optimized=True)

print("Holt-Winters Models Fitted:")
print(f"  Additive AIC: {hw_add_fitted.aic:.2f}")
print(f"  Multiplicative AIC: {hw_mul_fitted.aic:.2f}")

In [ ]:
# Forecast
forecast_hw_add = hw_add_fitted.forecast(steps=26)  # 26 weeks = 6 months
forecast_hw_mul = hw_mul_fitted.forecast(steps=26)
forecast_hw_index = pd.date_range(start=train_btc_weekly.index[-1] + pd.Timedelta(weeks=1), 
                                  periods=26, freq='W')

# Plot both
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Additive
axes[0].plot(train_btc_weekly.index[-52:], train_btc_weekly[-52:], 
            label='Training Data', color='blue', linewidth=2)
axes[0].plot(forecast_hw_index, forecast_hw_add, 
            label='HW Additive Forecast', color='red', linewidth=2, marker='o')
axes[0].set_title('Holt-Winters Additive Seasonal Model', fontweight='bold')
axes[0].set_ylabel('Bitcoin Price (USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Multiplicative
axes[1].plot(train_btc_weekly.index[-52:], train_btc_weekly[-52:], 
            label='Training Data', color='blue', linewidth=2)
axes[1].plot(forecast_hw_index, forecast_hw_mul, 
            label='HW Multiplicative Forecast', color='green', linewidth=2, marker='s')
axes[1].set_title('Holt-Winters Multiplicative Seasonal Model', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Bitcoin Price (USD)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- Holt-Winters captures **trend AND seasonality**
- **Additive**: Seasonal variations are constant in absolute terms
- **Multiplicative**: Seasonal variations are proportional to the level
- Choose based on whether seasonal fluctuations scale with the data level

---

# 📌 TOPIC 4.5: Vector Autoregression (VAR) Model

### Key Concepts:
VAR models **multiple time series simultaneously**, capturing interdependencies.

**VAR(p) for two variables:**

$$Y_{1,t} = c_1 + \phi_{11}Y_{1,t-1} + \phi_{12}Y_{2,t-1} + \epsilon_{1,t}$$

$$Y_{2,t} = c_2 + \phi_{21}Y_{1,t-1} + \phi_{22}Y_{2,t-1} + \epsilon_{2,t}$$

**Key Feature:** Each variable depends on:
- Its own past values
- Past values of other variables

**Granger Causality:** Tests if one variable helps predict another

In [ ]:
# Use Bitcoin and Gold returns (two series)
btc_ret = btc.pct_change().dropna() * 100
gold_ret = gold.pct_change().dropna() * 100

# Align dates
common_dates = btc_ret.index.intersection(gold_ret.index)
btc_ret_aligned = btc_ret.loc[common_dates]
gold_ret_aligned = gold_ret.loc[common_dates]

# Create bivariate dataset
data_var = pd.DataFrame({
    'BTC': btc_ret_aligned,
    'Gold': gold_ret_aligned
})

print(f"Bivariate dataset: {len(data_var)} observations")

In [ ]:
# Check stationarity
print("Stationarity Check (ADF Test):")
print("="*50)
for col in data_var.columns:
    adf = adfuller(data_var[col])
    status = '✓ Stationary' if adf[1] < 0.05 else '✗ Non-stationary'
    print(f"  {col}: p-value = {adf[1]:.4f} ({status})")

In [ ]:
# Fit VAR(2) model
train_var = data_var[:int(len(data_var)*0.8)]
var_model = VAR(train_var)
var_fitted = var_model.fit(maxlags=2)

print(f"VAR(2) Model Summary:")
print(f"  Number of observations: {var_fitted.nobs}")
print(f"  AIC: {var_fitted.aic:.2f}")
print(f"  BIC: {var_fitted.bic:.2f}")

In [ ]:
# Granger causality tests
print("\nGranger Causality Tests:")
print("="*50)
print("Testing if Gold Granger-causes Bitcoin:")
gc_test_gold_to_btc = grangercausalitytests(train_var[['BTC', 'Gold']], maxlag=2, verbose=False)
p_val_1 = gc_test_gold_to_btc[1][0]['ssr_ftest'][1]
p_val_2 = gc_test_gold_to_btc[2][0]['ssr_ftest'][1]
print(f"  Lag 1 p-value: {p_val_1:.4f} {'(Significant)' if p_val_1 < 0.05 else '(Not significant)'}")
print(f"  Lag 2 p-value: {p_val_2:.4f} {'(Significant)' if p_val_2 < 0.05 else '(Not significant)'}")

In [ ]:
# Forecast both series
forecast_var = var_fitted.forecast(train_var.values[-2:], steps=30)
forecast_var_df = pd.DataFrame(forecast_var, columns=['BTC', 'Gold'])
forecast_var_index = pd.date_range(start=train_var.index[-1] + pd.Timedelta(days=1), 
                                   periods=30, freq='D')

# Plot forecasts
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(train_var.index[-100:], train_var['BTC'][-100:], 
            label='Historical BTC Returns', color='blue')
axes[0].plot(forecast_var_index, forecast_var_df['BTC'], 
            label='VAR Forecast', color='red', linewidth=2, marker='o')
axes[0].set_title('Bitcoin Returns: VAR(2) Forecast', fontweight='bold')
axes[0].set_ylabel('Returns (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_var.index[-100:], train_var['Gold'][-100:], 
            label='Historical Gold Returns', color='green')
axes[1].plot(forecast_var_index, forecast_var_df['Gold'], 
            label='VAR Forecast', color='orange', linewidth=2, marker='s')
axes[1].set_title('Gold Returns: VAR(2) Forecast', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Returns (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- VAR models **multiple time series simultaneously**
- Each variable depends on its own lags AND other variables' lags
- **Granger causality** tests if past values of one variable help predict another
- Useful for analyzing interdependencies between related assets

---

# 📌 TOPIC 4.6: ARCH Effect Detection

### Key Concepts:
**ARCH effects** indicate **volatility clustering** - periods of high volatility tend to cluster together.

**How to detect:**
1. Check ACF of **squared returns** (should show significant lags)
2. Perform **ARCH LM test** (Lagrange Multiplier test)

**Interpretation:**
- Significant ACF in squared returns → Volatility is autocorrelated
- ARCH LM test p-value < 0.05 → ARCH effects present

In [ ]:
# Calculate Bitcoin returns
btc_returns = btc.pct_change().dropna() * 100

# Square the returns
btc_returns_sq = btc_returns ** 2

print(f"Bitcoin returns: {len(btc_returns)} observations")
print(f"Mean return: {btc_returns.mean():.4f}%")
print(f"Std deviation: {btc_returns.std():.4f}%")

In [ ]:
# Plot ACF of returns vs squared returns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(btc_returns, lags=30, ax=axes[0])
axes[0].set_title('ACF of Returns', fontweight='bold')

plot_acf(btc_returns_sq, lags=30, ax=axes[1])
axes[1].set_title('ACF of Squared Returns', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Perform ARCH LM test
arch_test = het_arch(btc_returns, nlags=10)
print("ARCH LM Test Results:")
print("="*50)
print(f"  Test Statistic: {arch_test[0]:.4f}")
print(f"  p-value: {arch_test[1]:.4f}")
print(f"  F-statistic: {arch_test[2]:.4f}")
print(f"  F p-value: {arch_test[3]:.4f}")

if arch_test[1] < 0.05:
    print("\n✓ Significant ARCH effects detected!")
    print("  → Volatility clustering is present")
    print("  → Consider using GARCH models")
else:
    print("\n✗ No significant ARCH effects")

### 📝 Key Takeaway:
- **Significant ACF in squared returns** = volatility clustering
- High volatility periods tend to be followed by high volatility
- Low volatility periods tend to be followed by low volatility
- When ARCH effects are present, use **GARCH models** to model volatility

---

# 📌 TOPIC 4.7: GARCH Model for Volatility

### Key Concepts:
**GARCH(1,1)** models time-varying volatility (conditional variance).

**GARCH(1,1) Formula:**
$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

Where:
- $\sigma_t^2$ = Conditional variance at time t
- $\omega$ = Constant (baseline variance)
- $\alpha$ = Weight of past squared shock (ARCH term)
- $\beta$ = Weight of past variance (GARCH term)
- $\alpha + \beta < 1$ ensures stationarity

In [ ]:
# Install arch package if needed
try:
    from arch import arch_model
    print("✓ arch package available")
except:
    print("Installing arch package...")
    !pip install arch --quiet
    from arch import arch_model
    print("✓ arch package installed")

In [ ]:
# Use Bitcoin returns
train_btc_ret = btc_returns[:int(len(btc_returns)*0.8)]

# Fit GARCH(1,1) model
garch_model = arch_model(train_btc_ret, vol='Garch', p=1, q=1)
garch_fitted = garch_model.fit(disp='off')

print("GARCH(1,1) Model Parameters:")
print("="*50)
print(f"  ω (omega): {garch_fitted.params['omega']:.6f}")
print(f"  α (alpha): {garch_fitted.params['alpha[1]']:.4f}")
print(f"  β (beta): {garch_fitted.params['beta[1]']:.4f}")
print(f"\n  α + β = {garch_fitted.params['alpha[1]'] + garch_fitted.params['beta[1]']:.4f}")
print("  (α + β < 1 ensures stationarity)")

In [ ]:
# Plot conditional volatility
conditional_vol = garch_fitted.conditional_volatility

plt.figure(figsize=(14, 6))
plt.plot(train_btc_ret.index, train_btc_ret, 
        label='Returns', color='blue', alpha=0.5, linewidth=1)
plt.plot(conditional_vol.index, conditional_vol, 
        label='Conditional Volatility (σₜ)', color='red', linewidth=2)
plt.plot(conditional_vol.index, -conditional_vol, 
        color='red', linewidth=2, alpha=0.5)
plt.title('Bitcoin Returns and GARCH(1,1) Conditional Volatility', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Returns (%) / Volatility')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Forecast volatility for next 30 days
forecast_garch = garch_fitted.forecast(horizon=30)
forecast_variance = forecast_garch.variance.iloc[-1].values
forecast_volatility = np.sqrt(forecast_variance)

plt.figure(figsize=(14, 5))
plt.plot(range(1, 31), forecast_volatility, marker='o', color='red', linewidth=2, markersize=6)
plt.title('GARCH(1,1) Volatility Forecast: Next 30 Days', fontsize=14, fontweight='bold')
plt.xlabel('Days Ahead')
plt.ylabel('Forecasted Volatility (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nVolatility Forecast Summary:")
print(f"  1-day ahead: {forecast_volatility[0]:.2f}%")
print(f"  30-day ahead: {forecast_volatility[29]:.2f}%")

### 📝 Key Takeaway:
- GARCH models **time-varying volatility** in financial returns
- Captures **volatility clustering**: high volatility follows high volatility
- $\alpha$ measures reaction to past shocks; $\beta$ measures persistence
- Volatility forecasts converge to long-run average as horizon increases

---

# 📌 TOPIC 4.8: Comprehensive Model Comparison

### Key Concepts:
Comparing multiple models helps select the best one for your data.

**Comparison Criteria:**
- **AIC/BIC**: In-sample fit (penalizes complexity)
- **RMSE**: Out-of-sample forecast accuracy
- **Visual inspection**: How well forecasts track actual data

**Best Practice:** Use multiple criteria, not just one!

In [ ]:
# Use MRF data
train_mrf_comp = mrf[:int(len(mrf)*0.8)]
test_mrf_comp = mrf[int(len(mrf)*0.8):]

# Define models to compare
models = {
    'AR(2)': (2, 0, 0),
    'MA(2)': (0, 0, 2),
    'ARMA(2,2)': (2, 0, 2),
    'ARIMA(1,1,1)': (1, 1, 1),
    'ARIMA(2,1,2)': (2, 1, 2)
}

print(f"Training set: {len(train_mrf_comp)} observations")
print(f"Test set: {len(test_mrf_comp)} observations")
print(f"\nModels to compare: {list(models.keys())}")

In [ ]:
# Fit all models and calculate metrics
results_comp = []

for name, order in models.items():
    try:
        model = ARIMA(train_mrf_comp, order=order)
        fitted = model.fit()
        
        # Forecast
        forecast = fitted.forecast(steps=len(test_mrf_comp))
        
        # Calculate RMSE on test set
        rmse = np.sqrt(np.mean((forecast.values - test_mrf_comp.values)**2))
        
        results_comp.append({
            'Model': name,
            'AIC': fitted.aic,
            'BIC': fitted.bic,
            'Test RMSE': rmse
        })
    except Exception as e:
        print(f"  Failed to fit {name}: {e}")

# Create comparison table
comparison_df = pd.DataFrame(results_comp)
comparison_df = comparison_df.sort_values('AIC')

print("\nModel Comparison Table:")
print("="*60)
print(comparison_df.to_string(index=False))

In [ ]:
# Find best models by each criterion
print(f"\n✓ Best model by AIC: {comparison_df.iloc[0]['Model']}")
print(f"✓ Best model by BIC: {comparison_df.sort_values('BIC').iloc[0]['Model']}")
print(f"✓ Best model by Test RMSE: {comparison_df.sort_values('Test RMSE').iloc[0]['Model']}")

In [ ]:
# Plot all forecasts together
plt.figure(figsize=(14, 7))
plt.plot(train_mrf_comp.index[-100:], train_mrf_comp[-100:], 
        label='Training Data', color='black', linewidth=2)
plt.plot(test_mrf_comp.index, test_mrf_comp, 
        label='Actual Test', color='blue', linewidth=2, alpha=0.7)

colors = ['red', 'green', 'orange', 'purple', 'brown']
for i, (name, order) in enumerate(models.items()):
    try:
        model = ARIMA(train_mrf_comp, order=order)
        fitted = model.fit()
        forecast = fitted.forecast(steps=len(test_mrf_comp))
        plt.plot(test_mrf_comp.index, forecast, 
                label=f'{name} Forecast', color=colors[i], 
                linewidth=1.5, linestyle='--', alpha=0.8)
    except:
        pass

plt.title('Model Comparison: All Forecasts', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('MRF Price (INR)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📝 Key Takeaway:
- Different models can produce **different forecasts**
- Use **multiple criteria** (AIC, BIC, RMSE) to select the best model
- **AIC/BIC** measure in-sample fit; **RMSE** measures out-of-sample accuracy
- The "best" model may differ depending on the criterion used

---

# ✅ MODULE 4 COMPLETE!

## Summary of Key Concepts:

| Topic | Model | Key Feature |
|-------|-------|-------------|
| 4.1 | SARIMA | Captures both trend and seasonality |
| 4.2 | Simple Exp. Smoothing | Weighted average with α parameter |
| 4.3 | Holt's Method | Adds linear trend component |
| 4.4 | Holt-Winters | Adds seasonal component (additive/multiplicative) |
| 4.5 | VAR | Models multiple series simultaneously |
| 4.6 | ARCH Detection | Identifies volatility clustering |
| 4.7 | GARCH | Models time-varying volatility |
| 4.8 | Model Comparison | Use multiple criteria to select best model |

---

**Next:** Module 5 - Integration & Practice